# Stock Price Prediction Model

This notebook covers the end-to-end process of predicting stock prices using historical data and machine learning.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

## 2. Data Acquisition
Fetch historical stock data using `yfinance`.

In [ ]:
# Define the ticker symbol and time range
ticker = 'AAPL'
start_date = '2015-01-01'
end_date = '2024-01-01'

# Fetch the data
print(f"Fetching data for {ticker}...")
df = yf.download(ticker, start=start_date, end=end_date)

# Display the first few rows and basic info
display(df.head())
print("\nDataset Info:")
display(df.info())

# Save raw data to CSV
data_path = f'../data/{ticker}_historical_data.csv'
df.to_csv(data_path)
print(f"\nData saved to {data_path}")

## 3. Exploratory Data Analysis (EDA)
Visualize trends and patterns.

In [ ]:
# Set style for seaborn
sns.set_theme(style='darkgrid')

# 1. Plot Closing Price History (Static)
plt.figure(figsize=(14, 7))
plt.plot(df.index, df['Close'], label='Close Price')
plt.title(f'{ticker} Closing Price History')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.show()

# 2. Interactive Candlestick Chart with Plotly
fig = go.Figure(data=[go.Candlestick(x=df.index,
                open=df['Open'],
                high=df['High'],
                low=df['Low'],
                close=df['Close')])
fig.update_layout(title=f'{ticker} Interactive Candlestick Chart',
                  yaxis_title='Price (USD)',
                  xaxis_rangeslider_visible=False)
fig.show()

# 3. Volume over Time
plt.figure(figsize=(14, 5))
plt.bar(df.index, df['Volume'], color='blue', alpha=0.5)
plt.title(f'{ticker} Trading Volume')
plt.xlabel('Date')
plt.ylabel('Volume')
plt.show()

## 4. Feature Engineering
Calculate technical indicators (SMA, RSI, etc.).

In [ ]:
def calculate_rsi(data, window=14):
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

# Calculate Moving Averages
df['SMA_20'] = df['Close'].rolling(window=20).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean()
df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()

# Calculate RSI
df['RSI_14'] = calculate_rsi(df)

# Calculate Daily Returns & Volatility
df['Daily_Return'] = df['Close'].pct_change()
df['Volatility_20'] = df['Daily_Return'].rolling(window=20).std()

# Define Target Variable (Next day's closing price)
df['Target_Next_Close'] = df['Close'].shift(-1)

# Drop NaN values caused by rolling windows and shifting
df.dropna(inplace=True)

print("Feature Engineering Complete.")
print(f"New dataset shape: {df.shape}")
display(df[['Close', 'SMA_20', 'RSI_14', 'Target_Next_Close']].head())

## 5. Modeling & Evaluation
Train the Random Forest model and evaluate its performance.

In [ ]:
# 1. Define Features (X) and Target (y)
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_20', 'SMA_50', 'EMA_20', 'RSI_14', 'Daily_Return', 'Volatility_20']
X = df[features]
y = df['Target_Next_Close']

# 2. Train-Test Split (Chronological, NO shuffling for time series)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 3. Initialize and Train the Model
print("Training Random Forest Model...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 4. Make Predictions
predictions = rf_model.predict(X_test)

# 5. Evaluate Performance
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
print(f"\nModel Performance on Test Set:")
print(f"Mean Absolute Error (MAE): ")
print(f"Root Mean Squared Error (RMSE): ")

# 6. Visualize Predictions vs Actual
plt.figure(figsize=(14, 7))
plt.plot(y_test.index, y_test.values, label='Actual Next Day Close Price', color='blue')
plt.plot(y_test.index, predictions, label='Predicted Next Day Close Price', color='red', alpha=0.7)
plt.title(f'{ticker} Price Prediction - Random Forest')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.show()